In [1]:
print("hello world")

hello world


In [3]:
import os

print(os.getcwd())

c:\Users\akolk\OneDrive\Documents\customer-churn-prediction\notebooks


In [4]:
import os

print("DATA folder:")
print(os.listdir("../data"))

DATA folder:
['processed', 'raw']


In [5]:
print("PROCESSED folder:")
print(os.listdir("../data/processed"))

PROCESSED folder:
[]


In [6]:
print("RAW folder:")
print(os.listdir("../data/raw"))

RAW folder:
['WA_Fn-UseC_-Telco-Customer-Churn.csv']


In [7]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

print(df.shape)

(7043, 21)


In [8]:
df["TotalCharges"] = df["TotalCharges"].replace(" ", "0")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])

print(df["TotalCharges"].dtype)

float64


In [9]:
df.to_csv(
    "../data/processed/eda_dataset.csv",
    index=False
)

print("EDA dataset saved successfully!")

EDA dataset saved successfully!


In [10]:
import os

print(
    os.path.exists(
        "../data/processed/eda_dataset.csv"
    )
)

True


In [11]:
import sqlite3

connection = sqlite3.connect("../data/customer_churn.db")

df.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

print("SQLite database created successfully!")

SQLite database created successfully!


In [12]:
cursor = connection.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")

print(cursor.fetchall())

[('customers',)]


In [13]:
query = """
SELECT COUNT(*) AS total_customers
FROM customers;
"""

result = pd.read_sql_query(query, connection)

print(result)

   total_customers
0             7043


In [14]:
query = """
SELECT COUNT(*) AS churned_customers
FROM customers
WHERE Churn = 'Yes';
"""

result = pd.read_sql_query(query, connection)

print(result)

   churned_customers
0               1869


In [15]:
query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate
FROM customers;
"""

result = pd.read_sql_query(query, connection)

print(result)

   total_customers  churned_customers  churn_rate
0             7043               1869       26.54


In [16]:
query = """
SELECT
    Contract,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC;
"""

result = pd.read_sql_query(query, connection)

print(result)

         Contract  total_customers  churned_customers  churn_rate
0  Month-to-month             3875               1655       42.71
1        One year             1473                166       11.27
2        Two year             1695                 48        2.83


In [17]:
query = """
SELECT
    PaymentMethod,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate DESC;
"""

result = pd.read_sql_query(query, connection)

print(result)

               PaymentMethod  total_customers  churned_customers  churn_rate
0           Electronic check             2365               1071       45.29
1               Mailed check             1612                308       19.11
2  Bank transfer (automatic)             1544                258       16.71
3    Credit card (automatic)             1522                232       15.24


In [18]:
query = """
SELECT
    CASE
        WHEN tenure <= 12 THEN 'New (0-12 months)'
        WHEN tenure <= 24 THEN 'Early (13-24 months)'
        WHEN tenure <= 48 THEN 'Established (25-48 months)'
        ELSE 'Long-term (49-72 months)'
    END AS tenure_group,

    COUNT(*) AS total_customers,

    SUM(
        CASE
            WHEN Churn = 'Yes' THEN 1
            ELSE 0
        END
    ) AS churned_customers,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN Churn = 'Yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate

FROM customers

GROUP BY tenure_group

ORDER BY churn_rate DESC;
"""

result = pd.read_sql_query(query, connection)

print(result)

                 tenure_group  total_customers  churned_customers  churn_rate
0           New (0-12 months)             2186               1037       47.44
1        Early (13-24 months)             1024                294       28.71
2  Established (25-48 months)             1594                325       20.39
3    Long-term (49-72 months)             2239                213        9.51


In [19]:
query = """
SELECT
    InternetService,
    COUNT(*) AS total_customers,
    SUM(
        CASE
            WHEN Churn = 'Yes' THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN Churn = 'Yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY InternetService
ORDER BY churn_rate DESC;
"""

result = pd.read_sql_query(query, connection)

print(result)

  InternetService  total_customers  churned_customers  churn_rate
0     Fiber optic             3096               1297       41.89
1             DSL             2421                459       18.96
2              No             1526                113        7.40


In [20]:
query = """
SELECT
    CASE
        WHEN MonthlyCharges > 70 THEN 'High Value'
        ELSE 'Standard Value'
    END AS customer_value_segment,

    COUNT(*) AS total_customers,

    SUM(
        CASE
            WHEN Churn = 'Yes' THEN 1
            ELSE 0
        END
    ) AS churned_customers,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN Churn = 'Yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate

FROM customers

GROUP BY customer_value_segment

ORDER BY churn_rate DESC;
"""

result = pd.read_sql_query(query, connection)

print(result)

  customer_value_segment  total_customers  churned_customers  churn_rate
0             High Value             3583               1267       35.36
1         Standard Value             3460                602       17.40


In [21]:
print(df.columns.tolist())

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [22]:
ml_df = df.copy()

print(ml_df.shape)

(7043, 21)


In [23]:
ml_df = ml_df.drop(columns=["customerID"])

print(ml_df.shape)

(7043, 20)


In [24]:
X = ml_df.drop(columns=["Churn"])
y = ml_df["Churn"]

In [25]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 19)
y shape: (7043,)


In [26]:
y = y.map({
    "No": 0,
    "Yes": 1
})

In [27]:
print(y.value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [28]:
X = ml_df.drop(columns=["Churn"])
y = ml_df["Churn"]

y = y.map({
    "No": 0,
    "Yes": 1
})

print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.value_counts())

X shape: (7043, 19)
y shape: (7043,)
Churn
0    5174
1    1869
Name: count, dtype: int64


In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [31]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)


In [32]:
print("Training churn distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True))

Training churn distribution:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Testing churn distribution:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64
